In [1]:
import pandas as pd
import numpy as np
import torch as th

from fragnnet.utils.formula_utils import ELEMENTS
from fragnnet.utils.proc_utils import filter_spec_mol
from fragnnet.baselines.graff.data_utils import fit_ann_library
from fragnnet.utils.data_utils import formula_to_mass

In [2]:

spec_df_fp = "../data/proc/nist20/spec_df.pkl"
mol_df_fp = "../data/proc/nist20/mol_df.pkl"
ann_df_fp = "../data/proc/nist20/ann_df.pkl"

spec_df = pd.read_pickle(spec_df_fp)
mol_df = pd.read_pickle(mol_df_fp)
ann_df = pd.read_pickle(ann_df_fp)

library_size  = 10000
max_isotope = 2
debug = True
huge_mz_diff_threshold = 0.1
big_mz_diff_ppm_threshold = 1e-5

In [3]:
# check split stuff
split_dp = "../data/split/nist20_d4_wr00_inchikey"
train_ids = pd.read_csv(f"{split_dp}/train_ids.csv")["spec_id"].to_numpy()
val_ids = pd.read_csv(f"{split_dp}/val_ids.csv")["spec_id"].to_numpy()
test_ids = pd.read_csv(f"{split_dp}/test_ids.csv")["spec_id"].to_numpy()
all_ids = np.concatenate([train_ids,val_ids,test_ids],axis=0)

# check train split
train_spec_df = spec_df[spec_df["spec_id"].isin(train_ids)]
# train_library_df = fit_ann_library(train_spec_df, ann_df, k, max_isotope)

# check val split
val_spec_df = spec_df[spec_df["spec_id"].isin(val_ids)]
# val_library_df = fit_ann_library(val_spec_df, ann_df, k, max_isotope)

# check all split
all_spec_df = spec_df[spec_df["spec_id"].isin(all_ids)].reset_index(drop=True)
# all_library_df = fit_ann_library(all_spec_df, ann_df, k, max_isotope)

In [4]:
ann_df.columns

Index(['dset_spec_id', 'mol_id', 'dset', 'spec_id', 'prec_type', 'formula',
       'ann_peak_mzs', 'ann_products', 'ann_losses', 'ann_isotopes',
       'ann_exact_mzs'],
      dtype='object')

In [5]:
orig_spec_ids = all_spec_df[["dset","dset_spec_id","spec_id"]].copy()
ann_spec_df = all_spec_df[all_spec_df["spec_id"].isin(ann_df["spec_id"])]
ann_spec_df = ann_spec_df[["spec_id","peaks"]]
# spec_df.loc[:,"peaks"] = spec_df["peaks"].apply(l1_normalize_peaks)
ann_spec_df = ann_spec_df.explode("peaks")
ann_spec_df[["mzs","ints"]] = pd.DataFrame(ann_spec_df["peaks"].tolist(), index=ann_spec_df.index)
ann_spec_df = ann_spec_df.drop(columns=["peaks"])
total_ints = ann_spec_df[["spec_id","ints"]].groupby("spec_id")["ints"].sum().reset_index().rename(columns={"ints": "total_ints"})
ann_spec_df = ann_spec_df.merge(total_ints, on="spec_id", how="inner")
ann_spec_df["ints"] = ann_spec_df["ints"] / ann_spec_df["total_ints"]
ann_spec_df = ann_spec_df.drop(columns=["total_ints"])

# select which (unmerged) spectra to use
# map mzs to (unmerged) intensities
ann_ann_df = ann_df[ann_df["spec_id"].isin(ann_spec_df["spec_id"])]
ann_ann_df = ann_ann_df[["spec_id","ann_peak_mzs","ann_products","ann_losses","ann_isotopes","ann_exact_mzs"]]
ann_ann_df = ann_ann_df.rename(columns={
    "ann_peak_mzs":"peak_mzs",
    "ann_products":"products",
    "ann_losses":"losses",
    "ann_isotopes":"isotopes",
    "ann_exact_mzs":"exact_mzs"})
ann_ann_df["joint"] = ann_ann_df.apply(lambda row: list(zip(row["peak_mzs"], row["products"], row["losses"], row["isotopes"], row["exact_mzs"])), axis=1)
ann_ann_df = ann_ann_df.drop(columns=["peak_mzs","products","losses","isotopes","exact_mzs"])
ann_ann_df = ann_ann_df.explode("joint")
ann_ann_df[["peak_mzs","products","losses","isotopes","exact_mzs"]] = pd.DataFrame(ann_ann_df["joint"].tolist(), index=ann_ann_df.index)
ann_ann_df = ann_ann_df.drop(columns=["joint"])
ann_ann_df["peak_mzs"] = ann_ann_df["peak_mzs"].astype("float64")
ann_ann_df["isotopes"] = ann_ann_df["isotopes"].astype("int64")
ann_ann_df["exact_mzs"] = ann_ann_df["exact_mzs"].astype("float64")
ann_ann_df = ann_ann_df.merge(ann_spec_df[["spec_id","mzs","ints"]].rename(columns={"mzs":"peak_mzs"}),on=["spec_id","peak_mzs"],how="inner")

In [6]:
# verify mz diffs
mz_diffs = ann_ann_df[["spec_id","peak_mzs","exact_mzs"]].copy()
mz_diffs["mz_diffs"] = (mz_diffs["peak_mzs"] - mz_diffs["exact_mzs"]).abs()
mz_diffs["mz_diffs_ppm"] = mz_diffs["mz_diffs"].to_numpy() / np.maximum(mz_diffs["peak_mzs"].to_numpy(), 200.)
if debug:
    print("> distribution of mzs diff (abs):")
    print(mz_diffs["mz_diffs"].describe())
    print("> distribution of mzs diff (ppm):")
    print(mz_diffs["mz_diffs_ppm"].describe())
huge_mz_diffs = mz_diffs[mz_diffs["mz_diffs"] > huge_mz_diff_threshold]
big_mz_diffs = mz_diffs[mz_diffs["mz_diffs_ppm"] > big_mz_diff_ppm_threshold]
huge_mz_diff_ids = huge_mz_diffs["spec_id"].unique()
big_mz_diff_ids = big_mz_diffs["spec_id"].unique()
if huge_mz_diff_ids.shape[0] > 0:
    print(f"> warning: dropping spectra with huge mz diffs: {huge_mz_diffs.shape[0]} annotations, {huge_mz_diff_ids.shape[0]} spectra")
    ann_ann_df = ann_ann_df[~(ann_ann_df["spec_id"].isin(huge_mz_diff_ids))]
if big_mz_diff_ids.shape[0] > 0:
    print(f"> warning: dropping annotations with big mz diffs: {big_mz_diffs.shape[0]} annotations, {big_mz_diff_ids.shape[0]} spectra")
    ann_ann_df = ann_ann_df[~(ann_ann_df.index.isin(big_mz_diffs.index))]

# verify isotopes
if debug:
    print("> distribution of isotopes:")
    print(ann_ann_df["isotopes"].value_counts())
invalid_isotopes = (ann_ann_df["isotopes"] > max_isotope) | (ann_ann_df["isotopes"] < 0)
if invalid_isotopes.any():
    print(f"> warning: dropping invalid isotopes: {invalid_isotopes.sum()} annotations")
    ann_ann_df = ann_ann_df[~invalid_isotopes]

# evenly divide intensity for each annotation that maps to the same bin
ann_counts = ann_ann_df[["spec_id","peak_mzs","products"]].groupby(["spec_id","peak_mzs"])["products"].count().reset_index().rename(columns={"products":"counts"})
ann_ann_df = ann_ann_df.merge(ann_counts, on=["spec_id","peak_mzs"], how="inner")
ann_ann_df["ints"] = ann_ann_df["ints"] / ann_ann_df["counts"]
ann_ann_df = ann_ann_df.drop(columns=["counts"])

# check fractions
ann_frac = ann_ann_df[["spec_id","ints"]].groupby(by=["spec_id"])["ints"].sum().reset_index()
ann_frac = ann_frac.merge(pd.DataFrame({"spec_id":orig_spec_ids["spec_id"]}),on="spec_id",how="right").fillna(0.)
iso_frac = ann_ann_df[ann_ann_df["isotopes"]>0][["spec_id","ints"]].groupby(by=["spec_id"])["ints"].sum().reset_index()
iso_frac = iso_frac.merge(pd.DataFrame({"spec_id":orig_spec_ids["spec_id"]}),on="spec_id",how="right").fillna(0.)
if debug:
    print("> distribution of annotated intensity fraction:")
    print(ann_frac["ints"].describe())
    print("> distribution of isotope intensity fraction:")
    print(iso_frac["ints"].describe())

# aggregate intensity by annotation
product_df = ann_ann_df[["products","ints"]].groupby("products")["ints"].sum().reset_index()
product_df["kind"] = "product"
product_df = product_df.rename(columns={"products":"formula"})
loss_df = ann_ann_df[["losses","ints"]].groupby("losses")["ints"].sum().reset_index()
loss_df["kind"] = "loss"
loss_df = loss_df.rename(columns={"losses":"formula"})
library_df = pd.concat([product_df, loss_df], axis=0, ignore_index=True)

# take top-k
library_df = library_df.sort_values(by="ints", ascending=False)
assert library_df.shape[0] >= library_size, f"library size {library_df.shape[0]} < {library_size}"
library_df = library_df.head(library_size).reset_index()

# enumerate
library_df["ann_id"] = np.arange(library_df.shape[0])

# calculate mzs
library_df["mzs"] = library_df["formula"].apply(formula_to_mass)

> distribution of mzs diff (abs):
count    1.023802e+07
mean     1.445427e-03
std      2.478449e-02
min      7.999972e-09
25%      4.731600e-04
50%      7.190840e-04
75%      9.742560e-04
max      3.988504e+00
Name: mz_diffs, dtype: float64
> distribution of mzs diff (ppm):
count    1.023802e+07
mean     6.656045e-06
std      1.131292e-04
min      1.396540e-11
25%      2.225640e-06
50%      3.448580e-06
75%      4.627080e-06
max      1.994252e-02
Name: mz_diffs_ppm, dtype: float64
> warning: dropping spectra with huge mz diffs: 4707 annotations, 2591 spectra
> warning: dropping annotations with big mz diffs: 471919 annotations, 91523 spectra
> distribution of isotopes:
isotopes
0    9624219
1       5418
2         12
Name: count, dtype: int64
> distribution of annotated intensity fraction:
count    262319.000000
mean          0.954825
std           0.176601
min           0.000000
25%           0.982836
50%           0.997690
75%           1.000000
max           1.000000
Name: ints, dtyp

In [7]:
mz_diffs.columns

Index(['spec_id', 'peak_mzs', 'exact_mzs', 'mz_diffs', 'mz_diffs_ppm'], dtype='object')

In [ ]:
library_df.iloc[0]

index            141613
formula                
ints       44470.381147
kind               loss
ann_id                0
mzs                 0.0
Name: 0, dtype: object

In [9]:
library_df.iloc[1]["formula"]

'C7H7'